In [1]:
# Pass marks, set before the run.
# Each method's saved output is first rescored the results table's way, and must give the exact-label
# precision, recall and F1 its notebook saved, to within this much; any difference raises.
SAME_WITHIN = 1e-9

# 16 · Every HumSet method by HumBERT's measure

The results tables score every method by exact-label F1 over all labels pooled, on the 1,500-excerpt samples,
with each trained model's labels picked by one rule chosen on development excerpts (notebooks 14 and 15).
HumBERT's paper reports a different measure: macro F1 in each of HumSet's five published groups of labels, with a
threshold per label chosen on validation, so every label counts equally (notebook 12).

This notebook scores every row of both HumSet tables by that measure too, on the same samples, so the two can
be set side by side. It reads what notebooks 14 and 15 saved and makes no LLM calls; the combined method's
combiner is retrained on the laptop, as those notebooks did.

- **Trained, fine-tuned and combined methods** rank labels, so each label gets its own threshold, chosen on the
  same development excerpts the table chose its cut on. This is the measure exactly as notebooks 10 and 15
  apply it to the full test sets.
- **LLM methods** return labels without scores, so there is no threshold to choose: each label returned counts,
  with its parent.

As on the full test sets, every label in a group counts, so a label that no sample excerpt has scores zero for
every method. The sample is not the paper's test set, so these figures say how methods compare under the
paper's measure, not whether they match its printed figures.

In [2]:
import numpy as np
import pandas as pd

from tiered_taxonomy import benchmarks as bm
from tiered_taxonomy import experiment as ex
from tiered_taxonomy import like_for_like as lf
from tiered_taxonomy import new_technique as nt
from tiered_taxonomy import signals as sg
from tiered_taxonomy.config import data_root, load_config
from tiered_taxonomy.methods import combined as cb

root = data_root()
config = load_config()
seed = config["seed"]
words = config["text"]["input_words"]
limits, thresholds = config["trained"]["cut_limits"], config["trained"]["cut_thresholds"]
settings = cb.CombinedSettings(per_source=config["combined"]["per_source"], seed=seed)
plan = config["new_technique"]["humset"]
SAVED = {
    ex.HUMSET: ex.load_results("14_like_for_like")["humset"],
    ex.HUMSET_V2: ex.load_results("15_humset_v2"),
}
STACKS = {
    ex.HUMSET: ex.load_results("11_second_attempt")["humset"]["sources"],
    ex.HUMSET_V2: ex.load_results("15_humset_v2")["choice"]["chosen"],
}


def same_as_saved(name, scores, saved):
    for key in ("f1", "precision", "recall"):
        if abs(scores["exact"][key] - saved["exact"][key]) > SAME_WITHIN:
            raise ValueError(
                f"{name}: rescored exact {key} {scores['exact'][key]} is not the saved {saved['exact'][key]}"
            )

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Score every row both ways

For each version: load the sample and the development excerpts the table tuned on, exactly as notebooks 14
and 19 chose them; rescore each method the table's way and check it against the saved score; then score it by
the paper's measure.

In [3]:
def score_version(name):
    data = ex.load_humset(root, name)
    saved = SAVED[name]
    sample = ex.sample_humset(data, config["samples"]["humset"]["per_language"], seed)
    order = np.random.default_rng(seed).permutation(len(data.dev))
    raw = [data.dev[int(i)] for i in order[: plan["fit_items"] + plan["cut_items"]]]
    fit, cut = nt.with_all_tiers(raw[: plan["fit_items"]]), nt.with_all_tiers(raw[plan["fit_items"] :])
    tune = raw[plan["fit_items"] :]
    ids = [item.id for item in sample]
    out = {"llm": {}, "ranked": {}, "combined": {}}

    for method, entry in saved["llm"].items():
        labels = ex.load_predictions(root, data.name, method, ids)
        same_as_saved(method, ex.evaluate(data, sample, labels), entry["scores"])
        out["llm"][method] = lf.paper_scores_of_labels(sample, labels, data.taxonomy)

    dev = sg.load_sources(root, data.name, list(saved["ranked"]), "dev")
    test = sg.load_sources(root, data.name, list(saved["ranked"]), "test")
    for method, entry in saved["ranked"].items():
        merged = {**dev[method], **test[method]}
        tune_rankings = lf.source_rankings(merged, tune, method)
        sample_rankings = lf.source_rankings(merged, sample, method)
        made = lf.score_ranked(data, tune, tune_rankings, sample, sample_rankings, limits, thresholds)
        same_as_saved(method, made["scores"], entry["scores"])
        out["ranked"][method] = lf.paper_scores(tune, tune_rankings, sample, sample_rankings, data.taxonomy)

    names = STACKS[name]
    graph = cb.LabelGraph(data.taxonomy, [sorted(bm.gold_with_parents(item)) for item in data.train])
    stack = nt.Stack(
        names,
        fit,
        cut,
        nt.expand_sources(sg.load_sources(root, data.name, names, "dev")),
        nt.expand_sources(sg.load_sources(root, data.name, names, "test")),
        graph,
        words,
        settings,
    )
    tune_rankings, sample_rankings = lf.combined_rankings(stack, sample)
    exact_labels = {label for item in data.train for label in item.labels}
    made = lf.score_ranked(
        data,
        tune,
        lf.restrict(tune_rankings, exact_labels),
        sample,
        lf.restrict(sample_rankings, exact_labels),
        limits,
        thresholds,
    )
    same_as_saved("combined", made["scores"], saved["combined"]["combined"]["scores"])
    out["combined"]["combined"] = lf.paper_scores(tune, tune_rankings, sample, sample_rankings, data.taxonomy)

    present = set().union(*(bm.gold_with_parents(item) for item in sample))
    out["labels_not_in_sample"] = {
        group: sorted(labels - present) for group, labels in bm.humset_categories(data.taxonomy).items()
    }
    return out


results = {name: score_version(name) for name in (ex.HUMSET, ex.HUMSET_V2)}
print("every saved score reproduced on both versions")
for name, out in results.items():
    print(name, "labels no sample excerpt has:", {g: len(v) for g, v in out["labels_not_in_sample"].items()})

every saved score reproduced on both versions
humset labels no sample excerpt has: {'Sectors': 0, 'Pillars 1D': 0, 'Subpillars 1D': 0, 'Pillars 2D': 0, 'Subpillars 2D': 0}
humset_v2 labels no sample excerpt has: {'Sectors': 0, 'Pillars 1D': 0, 'Subpillars 1D': 1, 'Pillars 2D': 0, 'Subpillars 2D': 2}


In [4]:
table = pd.concat(
    {
        name: pd.DataFrame(
            {
                method: {
                    "paper's macro F1": scores["Mean"],
                    "exact-label F1": SAVED[name][family][method]["scores"]["exact"]["f1"],
                }
                for family in ("llm", "ranked", "combined")
                for method, scores in out[family].items()
            }
        ).T
        for name, out in results.items()
    },
    axis=1,
)
table.sort_values((ex.HUMSET_V2, "paper's macro F1"), ascending=False).round(3)

humset                       humset_v2  \
                          paper's macro F1 exact-label F1 paper's macro F1   
combined                             0.583          0.662            0.635   
classifier                           0.574          0.628            0.602   
humbert_recipe                       0.570          0.637            0.591   
knn                                  0.488          0.590            0.573   
omikuji                              0.487          0.579            0.551   
ki_fsprompt                          0.440          0.519            0.516   
xr_transformer                       0.549          0.642            0.486   
reranker                             0.476          0.561            0.468   
embedding_classifier                 0.496          0.573            0.452   
gliner                               0.511          0.565            0.443   
llm                                  0.550          0.594            0.426   
funnel_names                         0.339          0.365            0.337   
funnel_cards                         0.335          0.395            0.337   
select_all                           0.342          0.405            0.328   
funnel_cards_checked                 0.316          0.371            0.314   
funnel_cards_quote_filter            0.302          0.352            0.300   
walk_branching                       0.274          0.419            0.294   
pipeline                             0.295          0.374            0.273   
llm_ensemble                         0.256          0.295            0.273   
walk_strict                          0.220          0.342            0.261   
label_search                         0.235          0.110            0.235   
walk_anchored                        0.183          0.188            0.199   

                                          
                          exact-label F1  
combined                           0.687  
classifier                         0.648  
humbert_recipe                     0.659  
knn                                0.661  
omikuji                            0.641  
ki_fsprompt                        0.563  
xr_transformer                     0.686  
reranker                           0.513  
embedding_classifier               0.596  
gliner                             0.482  
llm                                0.482  
funnel_names                       0.348  
funnel_cards                       0.352  
select_all                         0.375  
funnel_cards_checked               0.327  
funnel_cards_quote_filter          0.302  
walk_branching                     0.396  
pipeline                           0.330  
llm_ensemble                       0.310  
walk_strict                        0.351  
label_search                       0.101  
walk_anchored                      0.215

In [5]:
ex.save_results("16_paper_measure", results)
print("saved")

saved
